# Installation
`sagea==0.3.1a7` and later versions support installation in different environments, including standard Python (>=3.12) environments and lightweight installation in JupyterLite environments. In web-based environments such as JupyterLite, some features are currently unavailable due to limited support for certain dependencies. This setup is intended for demonstration purposes only. For full functionality, please install and use sagea in a standard Python environment. Run the example notebook in a new tab.

In [ ]:
%pip install sagea==0.3.1a7

"""
When installing sagea in web-based environments such as JupyterLite, only a minimal set of basic dependencies related to numerical computation is included by default. Therefore, additional plotting libraries or related dependencies need to be installed manually for creating figures later.
"""
%pip install matplotlib
%pip install cartopy

# Load gravity products (SHCs)

In [ ]:
import sagea
import pathlib

# define paths of products
pathlist_l2 = list(pathlib.Path("./data/GRACE_L2_Products/").glob("ITSG-Grace2018_n60_2008-*.gfc"))
pathlist_l2.sort()

path_gif48 = pathlib.Path("./data/auxiliary/GIF48.60.gfc")

# load products as SHC instance
lmax = 60

shc = sagea.SHC.io.from_gfc(pathlist_l2, lmax=lmax, key="gfc")
shc_gif48 = sagea.SHC.io.from_gfc(path_gif48, lmax=lmax, key="gfc")

# deduct background
shc -= shc_gif48

# Filtering
Due to limitations in providing large amounts of data in the web-based environment, only the filtering step in the postprocessing is demonstrated here.

In [ ]:
shc_filtered = shc.filter.slidewindowSwenson2006(n=3, m=5, a=30, k=10, window_length_min=5)
shc_filtered.filter.gaussian(radius=300, inplace=True)

# Synthesis into EWH grid and show the spatial distribution

In [ ]:
import cartopy
cartopy.config["data_dir"] = "./data/cartopy_data"
# Since the web-based JupyterLite environment cannot access online download
# services, the necessary local terrain/coastline datasets for plotting are
# provided separately and configured explicitly in this section.

shc_ewh_unfiltered = shc.convert(from_type='Geopotential', to_type='EWH')  # EWH in unit [m]
shc_ewh_filtered = shc_filtered.convert(from_type='Geopotential', to_type='EWH')

grid_unfiltered = shc_ewh_unfiltered.synthesize.to_grid(grid_space=1)
grid_filtered = shc_ewh_filtered.synthesize.to_grid(grid_space=1)

for grid in [grid_unfiltered, grid_filtered]:
    grid.plot(
        index=3,
        vmin=-0.3, vmax=0.3,
        projection=cartopy.crs.Robinson(),
        gridlines=False,
        coastline=True
    )